# SESSION 3: NumPy Advanced (Part 3: Matrix Ops & Manipulation)
## Assignment Track 3: Data Science Foundations
**Case Study:** Apna Kirana Store Multi-Product Matrix Analytics & Markov Market Share Engine  
**Author:** Atharv Chaudhari (`ds-work`)

---

### Topics Covered:
1. **Mathematical operations:** Addition, subtraction, multiplication, division
2. **Element-wise vs. Matrix Multiplication:** The crucial distinction between `*` and `@` / `np.matmul()`
3. **Matrix operations:** `dot()`, `matmul()`, transpose (`.T`), and co-occurrence matrices
4. **Statistical reductions:** `mean`, `median`, `std` (`ddof=0`), `var`, and `sum` across axes
5. **Linear Algebra (`np.linalg`):** Determinant (`det()`), Matrix Inverse (`inv()`), and Eigendecomposition (`eig()`)
6. **Markov Chain Steady-State Analysis:** Transition probability matrices and stationary market shares
7. **Array Manipulation:** `reshape()`, `flatten()`, `ravel()`, `concatenate()`, `stack()`, and `split()`
8. **Memory Profiling:** View semantics (`ravel()`) vs Deep Copies (`flatten()`) using `np.shares_memory()`


---
## Task 3: Daily Revenue Engine (3 Vectorized Methods) & Item Co-Sales Matrix

**Objectives:**
- Regenerate `units_sold` (6 items $	imes$ 7 days) and `price_inr` (6 prices) exactly as in Session 1 using `np.random.seed(42)`.
- Compute the 7 daily revenue totals using **three distinct mathematical approaches**:
  1. `price_inr.dot(units_sold)`: Dot product between price vector and sales matrix.
  2. `np.matmul(price_inr, units_sold)`: Formal matrix multiplication.
  3. `(units_sold * price_inr.reshape(6, 1)).sum(axis=0)`: Element-wise broadcasting followed by vertical column summation.
- Verify that all three methods yield mathematically identical results using `np.allclose`.
- Construct the item co-sales matrix `item_cosales = units_sold @ units_sold.T` and inspect its shape.
- *Hint applied:* `*` is strictly element-wise multiplication; never use `*` for linear algebraic matrix multiplication.


In [1]:
import numpy as np

# Set random seed 42 to exactly reproduce Session 1 data
np.random.seed(42)

# Item categories: rice, atta, sugar, tea, oil, dal
items = ['Rice', 'Atta', 'Sugar', 'Tea', 'Oil', 'Dal']
days = ['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun']

# Regenerate units_sold (6 items x 7 days)
units_sold = np.random.randint(5, 60, size=(6, 7))

# price_inr vector (6 item unit prices)
price_inr = np.array([52, 44, 45, 380, 140, 120])

print("=== units_sold Matrix (6 items x 7 days) ===")
print(units_sold)
print("\n=== price_inr Vector (INR) ===")
print(price_inr)


=== units_sold Matrix (6 items x 7 days) ===
[[43 56 33 19 47 12 25]
 [43 23 27 15 15 28 57]
 [40 44 28  7 26 57  6]
 [28 48 34 42  6 25 37]
 [16 26 48 29 53 31 46]
 [32 20 19 51 55 48 59]]

=== price_inr Vector (INR) ===
[ 52  44  45 380 140 120]


In [2]:
# Method 1: Vector-Matrix Dot Product
rev_way1 = price_inr.dot(units_sold)

# Method 2: np.matmul (Matrix Multiplication)
rev_way2 = np.matmul(price_inr, units_sold)

# Method 3: Element-wise broadcasting with reshape(6, 1) and sum along axis 0
rev_way3 = (units_sold * price_inr.reshape(6, 1)).sum(axis=0)

print("=== Daily Revenues (Mon - Sun in INR) ===")
print(f"Method 1 (price_inr.dot(units_sold))              : {rev_way1}")
print(f"Method 2 (np.matmul(price_inr, units_sold))       : {rev_way2}")
print(f"Method 3 ((units_sold * price.reshape).sum(axis=0)): {rev_way3}")

# Numerical Equivalence Check
is_all_equal = np.allclose(rev_way1, rev_way2) and np.allclose(rev_way2, rev_way3)
print(f"\nAll three methods match identically: {is_all_equal}")


=== Daily Revenues (Mon - Sun in INR) ===
Method 1 (price_inr.dot(units_sold))              : [22648 30184 26084 28103 20574 24021 31658]
Method 2 (np.matmul(price_inr, units_sold))       : [22648 30184 26084 28103 20574 24021 31658]
Method 3 ((units_sold * price.reshape).sum(axis=0)): [22648 30184 26084 28103 20574 24021 31658]

All three methods match identically: True


In [3]:
# Compute Item Co-Sales Matrix: units_sold @ units_sold.T
item_cosales = units_sold @ units_sold.T

print("=== Item Co-Sales Matrix (item_cosales) ===")
print(f"Shape: {item_cosales.shape} (Symmetric 6x6 Matrix)\n")
print(item_cosales)

# Explanation of Co-Sales Matrix
print("\nKey Insight: Diagonal entries item_cosales[i, i] represent the sum of squared sales for item i.")
print("Off-diagonal entries item_cosales[i, j] represent dot products measuring co-occurrence strength between items.")


=== Item Co-Sales Matrix (item_cosales) ===
Shape: (6, 6) (Symmetric 6x6 Matrix)

[[ 9413  6779  7297  7319  8292  8728]
 [ 6779  7590  5921  6755  7302  8646]
 [ 7297  5921  8330  6281  6752  7569]
 [ 7319  6755  6281  8038  7341  8357]
 [ 8292  7302  6752  7341  9963 10540]
 [ 8728  8646  7569  8357 10540 13196]]

Key Insight: Diagonal entries item_cosales[i, i] represent the sum of squared sales for item i.
Off-diagonal entries item_cosales[i, j] represent dot products measuring co-occurrence strength between items.


---
## Task 4: Weekly Statistical Summary & Busiest Day Analysis

**Objectives:**
- Compute comprehensive summary statistics for each item across the 7 days:
  - `total`: `units_sold.sum(axis=1)`
  - `mean`: `units_sold.mean(axis=1)`
  - `median`: `np.median(units_sold, axis=1)`
  - `std`: Population standard deviation (`units_sold.std(axis=1, ddof=0)`)
  - `var`: Population variance (`units_sold.var(axis=1, ddof=0)`)
- Format and display a clean 6-row weekly summary table.
- Identify the day of the week with the highest total unit sales.
- *Hint applied:* `np.argmax` counts from 0, so add 1 to obtain human-readable day numbers (Day 1 through Day 7).
- *Constraint applied:* **Use NumPy only; do not convert to pandas.**


In [4]:
# Compute statistical aggregates across axis 1 (across 7 days per item)
item_totals = units_sold.sum(axis=1)
item_means = units_sold.mean(axis=1)
item_medians = np.median(units_sold, axis=1)
item_stds = units_sold.std(axis=1, ddof=0)  # ddof=0: population standard deviation
item_vars = units_sold.var(axis=1, ddof=0)

print("=" * 82)
print(f"{'Item':<8} | {'SKU':<5} | {'Total':<6} | {'Mean':<8} | {'Median':<8} | {'Std Dev':<9} | {'Variance':<10}")
print("=" * 82)
for i in range(len(items)):
    print(f"{items[i]:<8} | {101+i:<5} | {item_totals[i]:<6} | {item_means[i]:<8.2f} | {item_medians[i]:<8.1f} | {item_stds[i]:<9.2f} | {item_vars[i]:<10.2f}")
print("=" * 82)


Item     | SKU   | Total  | Mean     | Median   | Std Dev   | Variance  
Rice     | 101   | 235    | 33.57    | 33.0     | 14.75     | 217.67    
Atta     | 102   | 208    | 29.71    | 27.0     | 14.19     | 201.35    
Sugar    | 103   | 208    | 29.71    | 28.0     | 17.52     | 307.06    
Tea      | 104   | 220    | 31.43    | 34.0     | 12.67     | 160.53    
Oil      | 105   | 249    | 35.57    | 31.0     | 12.57     | 157.96    
Dal      | 106   | 284    | 40.57    | 48.0     | 15.46     | 239.10    


In [5]:
# Determine Busiest Day (Summing across axis 0)
daily_unit_totals = units_sold.sum(axis=0)

busiest_day_idx = np.argmax(daily_unit_totals)
busiest_day_num = busiest_day_idx + 1
busiest_day_name = days[busiest_day_idx]
max_units = daily_unit_totals[busiest_day_idx]

print("=== Daily Unit Sales Breakdown ===")
for d_idx, day_name in enumerate(days):
    print(f"  Day {d_idx+1} ({day_name}): {daily_unit_totals[d_idx]} units sold")

print(f"\nBusiest Day: Day {busiest_day_num} ({busiest_day_name}) with {max_units} total units sold!")


=== Daily Unit Sales Breakdown ===
  Day 1 (Mon): 202 units sold
  Day 2 (Tue): 217 units sold
  Day 3 (Wed): 189 units sold
  Day 4 (Thu): 163 units sold
  Day 5 (Fri): 202 units sold
  Day 6 (Sat): 201 units sold
  Day 7 (Sun): 230 units sold

Busiest Day: Day 7 (Sun) with 230 total units sold!


---
## Task 1: Brand-Switching Markov Matrix & Steady-State Market Share

**Objectives:**
- Construct the $3 \times 3$ brand-switching probability matrix `switch_p` for three competing tea brands (e.g., Tata Tea, Red Label, Wagh Bakri):
  $$P = \begin{bmatrix} 0.7 & 0.2 & 0.1 \\ 0.3 & 0.5 & 0.2 \\ 0.2 & 0.3 & 0.5 \end{bmatrix}$$
- Compute the determinant with `np.linalg.det(switch_p)`.
- Invert the matrix with `np.linalg.inv(switch_p)` and verify $P^{-1} P = I$ with `np.allclose`.
- Run `np.linalg.eig` on the transpose $P^T$ (`switch_p.T`) to compute stationary market share.
- Extract the eigenvector corresponding to eigenvalue $\lambda = 1$, take its real component (`.real`), and normalize it into `steady_share` such that $\sum \text{steady\_share} = 1.0$.
- *Hint applied:* Eigenvectors are stored as the columns of the second return value of `np.linalg.eig`; identify the target index with `np.argmin(np.abs(vals - 1))`.


In [6]:
# 1. Build 3x3 tea brand-switching Markov probability matrix
switch_p = np.array([
    [0.7, 0.2, 0.1],  # Brand A retention (70%), switches to B (20%), C (10%)
    [0.3, 0.5, 0.2],  # Brand B retention (50%), switches to A (30%), C (20%)
    [0.2, 0.3, 0.5]   # Brand C retention (50%), switches to A (20%), B (30%)
])

print("=== Brand-Switching Probability Matrix (P) ===")
print(switch_p)

# Verify stochastic matrix property: each row sums to 1.0
print(f"Row sums: {switch_p.sum(axis=1)}")


=== Brand-Switching Probability Matrix (P) ===
[[0.7 0.2 0.1]
 [0.3 0.5 0.2]
 [0.2 0.3 0.5]]
Row sums: [1. 1. 1.]


In [7]:
# 2. Compute Determinant & Matrix Inverse
det_switch_p = np.linalg.det(switch_p)
inv_switch_p = np.linalg.inv(switch_p)

# Verify inv(P) @ P == Identity Matrix
is_inverse_identity = np.allclose(inv_switch_p @ switch_p, np.eye(3))

print(f"Determinant of switch_p : {det_switch_p:.4f} (Non-singular, invertible)")
print("\nInverse Matrix (switch_p^-1):\n", np.round(inv_switch_p, 4))
print(f"\nVerification: np.linalg.inv(switch_p) @ switch_p == np.eye(3) -> {is_inverse_identity}")


Determinant of switch_p : 0.1100 (Non-singular, invertible)

Inverse Matrix (switch_p^-1):
 [[ 1.7273 -0.6364 -0.0909]
 [-1.      3.     -1.    ]
 [-0.0909 -1.5455  2.6364]]

Verification: np.linalg.inv(switch_p) @ switch_p == np.eye(3) -> True


In [8]:
# 3. Markov Stationary Distribution: Eigendecomposition on switch_p.T
# The stationary distribution pi satisfies: pi @ P = pi  <=>  P.T @ pi.T = 1.0 * pi.T
eigenvalues, eigenvectors = np.linalg.eig(switch_p.T)

print("=== Eigendecomposition of Transpose P^T ===")
print("Eigenvalues :", eigenvalues)

# Find the index of eigenvalue closest to 1.0
idx_1 = np.argmin(np.abs(eigenvalues - 1.0))
target_eigenvalue = eigenvalues[idx_1]

# Extract corresponding eigenvector column and take real part
raw_eigenvector = eigenvectors[:, idx_1].real

# Normalize so steady-state probabilities sum to 1.0
steady_share = raw_eigenvector / raw_eigenvector.sum()

print(f"\nEigenvalue 1 found at index : {idx_1} (Value: {target_eigenvalue.real:.4f})")
print(f"Steady-State Market Share   : {steady_share}")
print(f"Steady-State Sum Check      : {steady_share.sum():.6f}")

# Verify steady state condition: steady_share @ switch_p == steady_share
is_steady_state = np.allclose(steady_share @ switch_p, steady_share)
print(f"Verification: steady_share @ switch_p == steady_share -> {is_steady_state}")

print("\n--- Long-Term Tea Brand Equilibrium ---")
print(f"  Brand A (Tata Tea)   : {steady_share[0]*100:.2f}%")
print(f"  Brand B (Red Label)  : {steady_share[1]*100:.2f}%")
print(f"  Brand C (Wagh Bakri) : {steady_share[2]*100:.2f}%")


=== Eigendecomposition of Transpose P^T ===
Eigenvalues : [1.       +0.j 0.4618034+0.j 0.2381966+0.j]

Eigenvalue 1 found at index : 0 (Value: 1.0000)
Steady-State Market Share   : [0.46341463 0.31707317 0.2195122 ]
Steady-State Sum Check      : 1.000000
Verification: steady_share @ switch_p == steady_share -> True

--- Long-Term Tea Brand Equilibrium ---
  Brand A (Tata Tea)   : 46.34%
  Brand B (Red Label)  : 31.71%
  Brand C (Wagh Bakri) : 21.95%


---
## Task 2: Multi-Week Array Joining, Stacking, Splitting & Memory Profiling

**Objectives:**
- Generate a second week of sales: `week2 = np.random.randint(5, 60, size=(6, 7))`.
- Join the two weeks horizontally into `two_weeks` ($6 \times 14$) with `np.concatenate(..., axis=1)`.
- Build a 3D tensor `stacked` ($2 \times 6 \times 7$) with `np.stack(..., axis=0)`.
- Split `two_weeks` back into two equal 7-day halves with `np.split(two_weeks, 2, axis=1)`.
- Assert that both halves equal the original `units_sold` and `week2`.
- Memory investigation: Mutate an element of `two_weeks.ravel()` and `two_weeks.flatten()`, profiling with `np.shares_memory()` to observe view vs. copy semantics.


In [9]:
# 1. Generate Week 2 sales data
week2 = np.random.randint(5, 60, size=(6, 7))

# 2. Concatenate two weeks horizontally (axis=1) into shape (6, 14)
two_weeks = np.concatenate([units_sold, week2], axis=1)

# 3. Stack two weeks along a new batch dimension (axis=0) into shape (2, 6, 7)
stacked = np.stack([units_sold, week2], axis=0)

print("=== Array Joining & Stacking ===")
print(f"Week 1 shape     : {units_sold.shape}")
print(f"Week 2 shape     : {week2.shape}")
print(f"two_weeks shape  : {two_weeks.shape} (6 items x 14 consecutive days)")
print(f"stacked 3D shape : {stacked.shape} (2 weeks x 6 items x 7 days)")


=== Array Joining & Stacking ===
Week 1 shape     : (6, 7)
Week 2 shape     : (6, 7)
two_weeks shape  : (6, 14) (6 items x 14 consecutive days)
stacked 3D shape : (2, 6, 7) (2 weeks x 6 items x 7 days)


In [10]:
# 4. Split two_weeks back into two halves along axis 1
halves = np.split(two_weeks, 2, axis=1)
first_half, second_half = halves[0], halves[1]

# Assert that reconstructed halves match the original arrays exactly
assert np.array_equal(first_half, units_sold), "First half does not equal Week 1!"
assert np.array_equal(second_half, week2), "Second half does not equal Week 2!"

print("=== Split & Equality Assertion ===")
print(f"First half shape  : {first_half.shape}")
print(f"Second half shape : {second_half.shape}")
print("Assertions PASSED: Reconstructed halves identically equal units_sold and week2!")


=== Split & Equality Assertion ===
First half shape  : (6, 7)
Second half shape : (6, 7)
Assertions PASSED: Reconstructed halves identically equal units_sold and week2!


In [11]:
# 5. Memory Profiling: ravel() vs. flatten()
# Create a test copy of two_weeks for mutation analysis
two_weeks_test = two_weeks.copy()

# ravel() creates a 1D view whenever memory is contiguous
rav = two_weeks_test.ravel()

# flatten() always creates a deep, independent copy
flat = two_weeks_test.flatten()

# Check memory sharing
shares_with_ravel = np.shares_memory(two_weeks_test, rav)
shares_with_flatten = np.shares_memory(two_weeks_test, flat)

print("=== Memory Sharing Inspection ===")
print(f"np.shares_memory(two_weeks_test, rav)  : {shares_with_ravel} (rav is a VIEW)")
print(f"np.shares_memory(two_weeks_test, flat) : {shares_with_flatten} (flat is a COPY)")

# Mutation Experiment
original_first_elem = two_weeks_test[0, 0]
original_second_elem = two_weeks_test[0, 1]

# Mutate element in ravel
rav[0] = 999

# Mutate element in flatten
flat[1] = 888

print("\n=== Mutation Results ===")
print(f"Original two_weeks_test[0, 0] was {original_first_elem} -> After rav[0] = 999: {two_weeks_test[0, 0]} (MUTATED!)")
print(f"Original two_weeks_test[0, 1] was {original_second_elem} -> After flat[1] = 888: {two_weeks_test[0, 1]} (UNCHANGED!)")


=== Memory Sharing Inspection ===
np.shares_memory(two_weeks_test, rav)  : True (rav is a VIEW)
np.shares_memory(two_weeks_test, flat) : False (flat is a COPY)

=== Mutation Results ===
Original two_weeks_test[0, 0] was 43 -> After rav[0] = 999: 999 (MUTATED!)
Original two_weeks_test[0, 1] was 56 -> After flat[1] = 888: 56 (UNCHANGED!)


---
## Summary of Key Learnings & Takeaways:

1. **Matrix Multiplication Operators:**
   - `*` performs Hadamard (element-wise) multiplication.
   - `@` and `np.matmul()` perform linear algebraic row-column matrix multiplication.
2. **Statistical Reductions:**
   - Always verify the reduction `axis`: `axis=0` aggregates vertically down columns (daily totals), while `axis=1` aggregates horizontally across rows (item totals).
   - In NumPy, `ddof=0` calculates population standard deviation / variance by default.
3. **Markov Chains in Data Science:**
   - Transition matrices describe state-switching dynamics.
   - The stationary distribution is obtained from the left eigenvector of $P$ with eigenvalue 1 (or equivalently, the right eigenvector of $P^T$).
4. **Memory Dynamics:**
   - `np.concatenate` expands along an existing axis; `np.stack` creates a brand new axis.
   - `ravel()` returns a memory-efficient view whenever possible, while `flatten()` always allocates a new buffer.
   - Use `np.shares_memory()` to verify whether two NumPy objects share the same physical RAM.
